In [1]:
!pip install -q datasets pandas pillow

In [5]:
from datasets import load_dataset

# Hugging Face maintains auto-converted parquet branches for script-based datasets under the 'parquet' split of the repository
funsd = load_dataset("nielsr/funsd-layoutlmv3", "default", revision="refs/convert/parquet")
cord = load_dataset("nielsr/cord-layoutlmv3", "default", revision="refs/convert/parquet")

print(funsd)
print(cord)

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

cord/cord-layoutlmv3-train.parquet: reconstructing file:   0%|          |  0.00B / 1.29GB            

cord/cord-layoutlmv3-train.parquet: downloading bytes:           |  0.00B            

cord/cord-layoutlmv3-validation.parquet: reconstructing file:   0%|          |  0.00B /  171MB            

cord/cord-layoutlmv3-validation.parquet: downloading bytes:           |  0.00B            

... (2 outputs left out)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'bboxes', 'ner_tags', 'image'],
        num_rows: 149
    })
    test: Dataset({
        features: ['id', 'tokens', 'bboxes', 'ner_tags', 'image'],
        num_rows: 50
    })
})
DatasetDict({
    train: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 100
    })
    test: Dataset({
        features: ['id', 'words', 'bboxes', 'ner_tags', 'image'],
        num_rows: 100
    })
})


In [8]:
for name, ds in [("FUNSD", funsd), ("CORD", cord)]:
    ex = ds["train"][0]
    print(name, "columns:", ds["train"].column_names)
    # Check dynamically which text column exists in the schema
    text_key = "tokens" if "tokens" in ex else "words"
    print("first 10 words:", ex[text_key][:10])
    print("first 3 boxes:", ex["bboxes"][:3])
    print("first 10 tags:", ex["ner_tags"][:10])
    print("image size:", ex["image"].size)
    print("---")

FUNSD columns: ['id', 'tokens', 'bboxes', 'ner_tags', 'image']
first 10 words: ['R&D', ':', 'Suggestion:', 'Date:', 'Licensee', 'Yes', 'No', '597005708', 'R&D', 'QUALITY']
first 3 boxes: [[383, 91, 493, 175], [287, 316, 295, 327], [124, 355, 221, 370]]
first 10 tags: [0, 3, 3, 3, 5, 3, 3, 0, 1, 2]
image size: (762, 1000)
---
CORD columns: ['id', 'words', 'bboxes', 'ner_tags', 'image']
first 10 words: ['x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x', 'x']
first 3 boxes: [[296, 288, 312, 300], [298, 310, 312, 322], [298, 330, 314, 342]]
first 10 tags: [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
image size: (864, 1296)
---


In [9]:
for name, ds in [("FUNSD", funsd), ("CORD", cord)]:
    boxes = [b for ex in ds["train"].select(range(50)) for b in ex["bboxes"]]
    print(name, "min:", min(min(b) for b in boxes), "max:", max(max(b) for b in boxes))

FUNSD min: 7 max: 992
CORD min: 0 max: 996


In [10]:
funsd_labels = funsd["train"].features["ner_tags"].feature.names
cord_labels = cord["train"].features["ner_tags"].feature.names
print(funsd_labels)
print(len(cord_labels), cord_labels[:10])

['O', 'B-HEADER', 'I-HEADER', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER']
61 ['O', 'B-MENU.NM', 'B-MENU.NUM', 'B-MENU.UNITPRICE', 'B-MENU.CNT', 'B-MENU.DISCOUNTPRICE', 'B-MENU.PRICE', 'B-MENU.ITEMSUBTOTAL', 'B-MENU.VATYN', 'B-MENU.ETC']


In [11]:
import os
for name, ds in [("funsd", funsd), ("cord", cord)]:
    os.makedirs(f"samples/{name}", exist_ok=True)
    for i in range(3):
        ds["test"][i]["image"].convert("RGB").save(f"samples/{name}/{name}_test_{i}.png")
!ls samples/*

samples/cord:
cord_test_0.png  cord_test_1.png  cord_test_2.png

samples/funsd:
funsd_test_0.png  funsd_test_1.png  funsd_test_2.png


In [12]:
!zip -r samples.zip samples

  adding: samples/ (stored 0%)
  adding: samples/cord/ (stored 0%)
  adding: samples/cord/cord_test_0.png (deflated 0%)
  adding: samples/cord/cord_test_2.png (deflated 0%)
  adding: samples/cord/cord_test_1.png (deflated 0%)
  adding: samples/funsd/ (stored 0%)
  adding: samples/funsd/funsd_test_2.png (deflated 10%)
  adding: samples/funsd/funsd_test_1.png (deflated 11%)
  adding: samples/funsd/funsd_test_0.png (deflated 9%)


In [15]:
def stats(ds):
    return {k: len(v) for k, v in ds.items()}

summary = f"""# Benchmarks Summary (Person 1)

## FUNSD (form understanding)
- Source: nielsr/funsd-layoutlmv3 on Hugging Face (original: guillaumejaume.github.io/FUNSD)
- Splits: {stats(funsd)}
- Columns: {funsd['train'].column_names}
- Labels ({len(funsd_labels)}): {funsd_labels}
- Boxes normalised to 0-1000: Yes (observed values range from 7 to 992)

## CORD (receipt understanding)
- Source: nielsr/cord-layoutlmv3 on Hugging Face (original: github.com/clovaai/cord)
- Splits: {stats(cord)}
- Columns: {cord['train'].column_names}
- Number of labels: {len(cord_labels)}
- Boxes normalised to 0-1000: Yes (observed values range from 0 to 996)

## Notes
- Difference from official split sizes (if any): None. Matches the Hugging Face canonical distribution.
- Problems faced while loading: Native dataset python loader scripts are deprecated and blocked by HF datasets package raising a RuntimeError. Solved by loading pre-converted parquet files from Hugging Face using the revision='refs/convert/parquet' branch.
"""
os.makedirs("docs", exist_ok=True)
with open("docs/benchmarks_summary.md", "w") as f:
    f.write(summary)
print(summary)

# Benchmarks Summary (Person 1)

## FUNSD (form understanding)
- Source: nielsr/funsd-layoutlmv3 on Hugging Face (original: guillaumejaume.github.io/FUNSD)
- Splits: {'train': 149, 'test': 50}
- Columns: ['id', 'tokens', 'bboxes', 'ner_tags', 'image']
- Labels (7): ['O', 'B-HEADER', 'I-HEADER', 'B-QUESTION', 'I-QUESTION', 'B-ANSWER', 'I-ANSWER']
- Boxes normalised to 0-1000: Yes (observed values range from 7 to 992)

## CORD (receipt understanding)
- Source: nielsr/cord-layoutlmv3 on Hugging Face (original: github.com/clovaai/cord)
- Splits: {'train': 800, 'validation': 100, 'test': 100}
- Columns: ['id', 'words', 'bboxes', 'ner_tags', 'image']
- Number of labels: 61
- Boxes normalised to 0-1000: Yes (observed values range from 0 to 996)

## Notes
- Difference from official split sizes (if any): None. Matches the Hugging Face canonical distribution.
- Problems faced while loading: Native dataset python loader scripts are deprecated and blocked by HF datasets package raising a RuntimeEr